<a href="https://colab.research.google.com/github/MaggieHDez/ClassFiles/blob/PLN/analisisTemasClustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install gensim nltk pyLDAvis

In [18]:
import gensim
from gensim import corpora
from gensim.models import LdaModel
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
from gensim.models import CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import warnings # Para ignorar las advertencias
import pandas as pd

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning, module='gensim')

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)  # <-- para un error reciente de word_tokenize
nltk.download('stopwords', quiet=True)


# ----------------------------
# Conjunto de documentos
# ----------------------------
documents = [
    # Política
    "El presidente anunció nuevas reformas durante la conferencia en el Congreso.",
    "Los partidos políticos debatieron sobre el futuro del sistema electoral.",
    "La oposición critica la gestión gubernamental en materia de seguridad.",
    "Miles de personas protestaron en la capital contra la nueva ley propuesta.",
    "Se firmó un acuerdo internacional para fomentar la cooperación política.",

    # Economía
    "El banco central decidió aumentar las tasas de interés para controlar la inflación.",
    "La economía del país creció un 3% durante el segundo trimestre del año.",
    "Las exportaciones de productos agrícolas aumentaron notablemente.",
    "El desempleo bajó por tercer mes consecutivo, según datos oficiales.",
    "Los inversores mostraron confianza ante las nuevas políticas económicas.",

    # Deportes
    "La selección nacional clasificó a la final del campeonato continental.",
    "El delantero estrella fue transferido al club europeo por una suma millonaria.",
    "Los fanáticos celebraron la victoria con una caravana en las calles.",
    "El equipo local perdió el partido decisivo por un gol en el último minuto.",
    "La liga anunció cambios en el reglamento para la próxima temporada.",

    # Tecnología
    "La empresa lanzó un nuevo smartphone con inteligencia artificial integrada.",
    "Se descubrió una vulnerabilidad crítica en el sistema operativo.",
    "La inversión en energías renovables incluye avances en baterías inteligentes.",
    "El uso de la automatización ha transformado la industria manufacturera.",
    "Expertos debaten sobre los riesgos éticos del desarrollo de la IA.",

    # Medio ambiente
    "El cambio climático está afectando los patrones de lluvia en la región.",
    "Un nuevo informe advierte sobre la pérdida acelerada de biodiversidad.",
    "Se implementaron miles de políticas públicas para reducir las emisiones de carbono.",
    "Miles de voluntarios participaron en una jornada de reforestación.",
    "Organizaciones internacionales exigen medidas urgentes contra la contaminación."
]


# ----------------------------
# Preprocesamiento simple
# ----------------------------
# Preprocesamiento simple
stop_words = set(stopwords.words('spanish'))
texts = [
    [word for word in word_tokenize(doc.lower()) if word.isalpha() and word not in stop_words]
    for doc in documents
]

# Crear diccionario y corpus
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

# Entrenar modelo LDA
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=5, random_state=42,
                     passes=20,          # era 1 por defecto
                     iterations=400,     # más iteraciones para tener convergencia
                     )

# Mostrar temas
for idx, topic in lda_model.print_topics():
    print(f"Tema {idx}:\n{topic}\n")


# ----------------------------
# Evaluación del modelo
# ----------------------------


# Perplejidad (mide qué tan bien el modelo predice nuevos datos)
perplexity = lda_model.log_perplexity(corpus)
print(f"📉 Perplejidad del modelo: {perplexity:.4f}")

# Coherencia (mide el grado de similitud semántica entre las N palabras principales de cada tema)
coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
coherence = coherence_model.get_coherence()
print(f"📈 Coherencia del modelo: {coherence:.4f}")


# ----------------------------
# Visualización con pyLDAvis
# ----------------------------

vis = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(vis)  # No necesitas enable_notebook()


pyLDAvis.save_html(vis, 'lda_output.html')


# ----------------------------
# Prueba con distintos valores de α, β y número de temas
# ----------------------------

alphas = ['symmetric', 'asymmetric', 0.5, 0.9, 1, 1.2]
betas = ['symmetric', None, 0.3, 0.5, 0.7, 0.9, 1, 1.2]
topics = [3, 5, 7]

results = []

for a in alphas:
    for b in betas:
        for t in topics:
            model = LdaModel(
                corpus=corpus,
                id2word=dictionary,
                num_topics=t,
                alpha=a,
                eta=b,
                passes=20,
                iterations=400,
                random_state=42
            )
            perplexity = model.log_perplexity(corpus)
            coherence_model = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v')
            coherence = coherence_model.get_coherence()

            results.append({
                'Alpha': a,
                'Beta': b,
                'Num_Topics': t,
                'Perplejidad': perplexity,
                'Coherencia': coherence
            })

# Mostrar resultados ordenados por coherencia
df_results = pd.DataFrame(results).sort_values(by='Coherencia', ascending=False)
print("\n📊 Resultados comparativos:")
display(df_results)

# Descarga de datos
df_results.to_csv('resultados_comparativos.csv', index=False)

# ----------------------------
# Modelo óptimo según las métricas
# ----------------------------

best_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=5,        # Número óptimo de temas
    alpha='asymmetric',  # Distribución de temas por documento
    eta='symmetric',     # Distribución de palabras por tema
    passes=20,
    iterations=400,
    random_state=42
)

# Mostrar los temas principales del modelo óptimo
print("\n🚀 Modelo óptimo:")
for idx, topic in best_model.print_topics():
    print(f"Tema {idx}:\n{topic}\n")

# -----------------------------
# Evaluación del modelo óptimo
# -----------------------------

# Perplejidad (mide qué tan bien el modelo predice nuevos datos)
perplexity = lda_model.log_perplexity(corpus)
print(f"📉 Perplejidad del modelo: {perplexity:.4f}")

# Coherencia (mide el grado de similitud semántica entre las N palabras principales de cada tema)
coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
coherence = coherence_model.get_coherence()
print(f"📈 Coherencia del modelo: {coherence:.4f}")

# ----------------------------
# Visualización interactiva con pyLDAvis
# ----------------------------
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

vis_best = gensimvis.prepare(best_model, corpus, dictionary)
pyLDAvis.display(vis_best)  # Si estás en Jupyter, se muestra directamente

# Guardar como HTML
pyLDAvis.save_html(vis_best, 'lda_best_model.html')
print("✅ Visualización guardada como 'lda_best_model.html'")



Tema 0:
0.022*"anunció" + 0.022*"nuevo" + 0.022*"club" + 0.022*"suma" + 0.022*"europeo" + 0.022*"millonaria" + 0.022*"estrella" + 0.022*"delantero" + 0.022*"transferido" + 0.022*"acelerada"

Tema 1:
0.019*"nuevas" + 0.019*"sistema" + 0.019*"aumentar" + 0.019*"central" + 0.019*"inflación" + 0.019*"interés" + 0.019*"banco" + 0.019*"decidió" + 0.019*"controlar" + 0.019*"tasas"

Tema 2:
0.039*"miles" + 0.022*"local" + 0.022*"decisivo" + 0.022*"minuto" + 0.022*"partido" + 0.022*"último" + 0.022*"perdió" + 0.022*"equipo" + 0.022*"gol" + 0.022*"nueva"

Tema 3:
0.020*"inteligentes" + 0.020*"inversión" + 0.020*"renovables" + 0.020*"baterías" + 0.020*"incluye" + 0.020*"energías" + 0.020*"avances" + 0.020*"campeonato" + 0.020*"final" + 0.020*"política"

Tema 4:
0.030*"políticas" + 0.016*"datos" + 0.016*"según" + 0.016*"bajó" + 0.016*"mes" + 0.016*"oficiales" + 0.016*"consecutivo" + 0.016*"desempleo" + 0.016*"tercer" + 0.016*"reducir"

📉 Perplejidad del modelo: -5.8623
📈 Coherencia del modelo: 0.3

,Alpha,Beta,Num_Topics,Perplejidad,Coherencia
143,1.2,1.2,7,-5.736163,0.499849
47,asymmetric,1.2,7,-5.601852,0.473931
134,1.2,0.7,7,-6.061674,0.470960
140,1.2,1,7,-5.826822,0.468499
44,asymmetric,1,7,-5.623613,0.464681
...,...,...,...,...,...
106,1,0.5,5,-5.991631,0.282187
85,0.9,0.7,5,-5.882952,0.280702
130,1.2,0.5,5,-6.030819,0.278320
109,1,0.7,5,-5.909283,0.262809



🚀 Modelo óptimo:
Tema 0:
0.012*"sistema" + 0.012*"políticas" + 0.012*"anunció" + 0.012*"nuevo" + 0.012*"miles" + 0.012*"europeo" + 0.012*"club" + 0.012*"suma" + 0.012*"renovables" + 0.012*"incluye"

Tema 1:
0.025*"nuevas" + 0.014*"partido" + 0.014*"equipo" + 0.014*"interés" + 0.014*"inflación" + 0.014*"gol" + 0.014*"banco" + 0.014*"último" + 0.014*"tasas" + 0.014*"aumentar"

Tema 2:
0.046*"miles" + 0.025*"nueva" + 0.025*"personas" + 0.025*"capital" + 0.025*"ley" + 0.025*"protestaron" + 0.025*"propuesta" + 0.025*"exigen" + 0.025*"organizaciones" + 0.025*"urgentes"

Tema 3:
0.007*"sistema" + 0.007*"anunció" + 0.007*"nuevo" + 0.007*"políticas" + 0.007*"miles" + 0.007*"exportaciones" + 0.007*"manufacturera" + 0.007*"aumentaron" + 0.007*"automatización" + 0.007*"transformado"

Tema 4:
0.032*"según" + 0.032*"bajó" + 0.032*"datos" + 0.032*"mes" + 0.032*"consecutivo" + 0.032*"oficiales" + 0.032*"desempleo" + 0.032*"tercer" + 0.005*"sistema" + 0.005*"anunció"

📉 Perplejidad del modelo: -5.8623